In [1]:
import numpy as np
from aeon.classification.interval_based import RSTSF
from sklearn.metrics import accuracy_score
from tscglue.models import LokyStackerV7, LokyStackerV7SoftFilterRidge, TSCGlue, LokyStackerV8Base, LokyStackerV8AutoBestStacking, LokyStackerV8AutoBestBase
from tscglue.old_models import StackerV4, LokyStackerV5, LokyStackerV5SoftRF, LokyStackerV6
from tscglue import utils, data_loader
import polars as pl

In [ ]:
dataset = "Worms"
# dataset = 'Car'
# dataset = 'HandOutlines'
dataset = 'Trace'
X_train, y_train, X_test, y_test = data_loader.load_fold(dataset, 0)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(181, 1, 900) (181,) (77, 1, 900) (77,)


In [3]:
#m = TSCGlue(random_state=492, n_repetitions=1, n_jobs=16, keep_features=True, verbose=2)
#m.fit(X_train, y_train)
#y_pred = m.predict(X_test)

In [4]:
m = LokyStackerV8Base(random_state=492, n_repetitions=1, n_jobs=16, keep_features=True, verbose=2)
m.fit(X_train, y_train)

[0.00s] Starting executor with 16 workers, run_dir=./tscglue_runs/c306521162404ddd
[0.00s] Saved X and y to disk in 0.00s
[0.49s] Computed QUANT features (181, 7987) (11.03 MB) in 0.4836s
[0.54s] Starting repetition 0
[2.51s] Computed MultiRocket features (181, 49728) (68.67 MB) in 1.9707s
[5.03s] Computed Hydra features (181, 7168) (9.90 MB) in 2.5174s
[13.34s] Computed RDST features (181, 30000) (41.43 MB) in 8.3131s
[13.35s] Starting training with 16 workers for 40 models
[15.28s] Trained multirockethydra-bestk-p-ridgecv_r0 in 1.9264s for f-2/r-0 (3.06 MB)
[15.35s] Trained multirockethydra-bestk-p-ridgecv_r0 in 1.9912s for f-4/r-0 (3.06 MB)
[15.42s] Trained multirockethydra-bestk-p-ridgecv_r0 in 2.0594s for f-0/r-0 (3.06 MB)
[15.44s] Trained multirockethydra-bestk-p-ridgecv_r0 in 2.0780s for f-3/r-0 (3.06 MB)
[15.47s] Trained multirockethydra-bestk-p-ridgecv_r0 in 2.1143s for f-5/r-0 (3.06 MB)
[15.51s] Trained multirockethydra-bestk-p-ridgecv_r0 in 2.1586s for f-1/r-0 (3.06 MB)
[16.

RuntimeError: Worker failed during training quant-etc_r0 fold 0: [Errno 2] No such file or directory: './tscglue_runs/c306521162404ddd/models/quant-etc_r_0_f_0.pkl'

In [ ]:
preds = m.predict_per_model(X_test)
tests_accs = []
for model_name, model_preds in preds.items():
    acc = accuracy_score(y_test, model_preds)
    tests_accs.append({"model": model_name, "test_accuracy": acc})
test_df = pl.DataFrame(tests_accs)

[0.00s] Starting prediction
[0.00s] Starting executor with 16 workers, run_dir=./tscglue_runs/4cdfa4b0e2534157
[3.68s] Computed QUANT features (77, 7987) (2.35 MB) in 3.682947s
[4.11s] Computed repetition 0 MultiRocket features (77, 49728) (14.61 MB) in 0.425264s
[6.58s] Computed repetition 0 Hydra features (77, 7168) (2.11 MB) in 2.473782s
[12.41s] Computed repetition 0 RDST features (77, 30000) (17.62 MB) in 5.823059s
[12.48s] Computed features for prediction
[12.53s] Feature arrays saved to mmap files
[12.53s] Starting prediction with 16 workers for 40 first-level models
[13.23s] Predicted rdst-p-ridgecv_r0 in 0.0448s
[13.24s] Predicted rdst-p-ridgecv_r0 in 0.0646s
[13.28s] Predicted rdst-p-ridgecv_r0 in 0.0663s
[13.28s] Predicted rdst-p-ridgecv_r0 in 0.0705s
[13.29s] Predicted rdst-p-ridgecv_r0 in 0.1023s
[13.30s] Predicted rdst-p-ridgecv_r0 in 0.1208s
[13.30s] Predicted rdst-p-ridgecv_r0 in 0.0726s
[13.35s] Predicted rdst-p-ridgecv_r0 in 0.1023s
[13.37s] Predicted quant-etc_r0 in 

In [ ]:
pl.DataFrame(m.summary())

model,level,oof_accuracy
str,i64,f64
"""multirockethydra-bestk-p-ridge…",0,0.756906
"""rdst-p-ridgecv_r0""",0,0.729282
"""rstsf_r0""",0,0.712707
"""quant-etc_r0""",0,0.701657
"""probability-ridgecv_r0""",1,0.740331


In [ ]:
pl.DataFrame(m.summary()).join(test_df, on="model").sort("test_accuracy", descending=True)

model,level,oof_accuracy,test_accuracy
str,i64,f64,f64
"""multirockethydra-bestk-p-ridge…",0,0.756906,0.805195
"""rstsf_r0""",0,0.712707,0.792208
"""quant-etc_r0""",0,0.701657,0.779221
"""rdst-p-ridgecv_r0""",0,0.729282,0.688312


In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.00s] Starting prediction
[0.00s] Starting executor with 16 workers, run_dir=./tscglue_runs/4cdfa4b0e2534157
[3.31s] Computed QUANT features (77, 7987) (2.35 MB) in 3.307783s
[3.73s] Computed repetition 0 MultiRocket features (77, 49728) (14.61 MB) in 0.424854s
[6.49s] Computed repetition 0 Hydra features (77, 7168) (2.11 MB) in 2.753385s
[12.12s] Computed repetition 0 RDST features (77, 30000) (17.62 MB) in 5.638013s
[12.25s] Computed features for prediction
[12.28s] Feature arrays saved to mmap files
[12.28s] Starting prediction with 16 workers for 40 first-level models
[15.70s] Predicted rdst-p-ridgecv_r0 in 0.0460s
[15.74s] Predicted rdst-p-ridgecv_r0 in 0.0787s
[15.76s] Predicted rdst-p-ridgecv_r0 in 0.0780s
[15.76s] Predicted rdst-p-ridgecv_r0 in 0.0954s
[15.76s] Predicted rdst-p-ridgecv_r0 in 0.0517s
[15.78s] Predicted rdst-p-ridgecv_r0 in 0.1230s
[15.79s] Predicted rdst-p-ridgecv_r0 in 0.1353s
[15.81s] Predicted quant-etc_r0 in 0.0264s
[15.85s] Predicted rdst-p-ridgecv_r0 in 

In [ ]:
F = m.get_features()
F.estimated_size('gb')

0.12795495241880417

In [ ]:
P = m.get_oof_predictions()
P.estimated_size('gb')

3.371387720108032e-05

In [ ]:
m = LokyStackerV6(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train, y_train)

[0.0000s] Starting fitting
Starting executor with 16 workers, run_dir=./tscglue_runs/3db5077ed32b4ba6
[2.7946s] Computed QUANT features (181, 7987) (11.03 MB) in 2.7905s
[2.7976s] Starting repetition 0
[4.3008s] Computed MultiRocket features (181, 49728) (68.67 MB) in 1.5031s
[8.0955s] Computed Hydra features (181, 7168) (9.90 MB) in 3.7942s
[23.6057s] Computed RDST features (181, 30000) (41.43 MB) in 15.5099s
[23.6111s] Starting training with 16 workers for 40 models


Process SpawnProcess-62:
Process SpawnProcess-51:
Process SpawnProcess-49:
Process SpawnProcess-61:
Process SpawnProcess-64:
Process SpawnProcess-57:
Process SpawnProcess-60:
Process SpawnProcess-53:
Process SpawnProcess-52:
Process SpawnProcess-55:
Process SpawnProcess-58:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/petelin/.local/share/uv/python/cpython-3.13.2-linux-x86_64-gnu/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/home/petelin/.local/share/uv/python/cpython-3.13.2-linux-x86_64-gnu/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/petelin/.local/share/uv/python/cpython-3.13.2-linux-x86_64-gnu/lib/python3.13/concurrent/futures/process.py", line 242, in _process_worker
    call_item = call_queue.get(block=True)
  File "/home/petel

Executor shutdown complete


KeyboardInterrupt: 

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

In [ ]:
m = LokyStackerV5(random_state=492, n_repetitions=1, n_jobs=16)
m.fit(X_train, y_train)

In [ ]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")